In [1]:
%env CUDA_VISIBLE_DEVICES=1

env: CUDA_VISIBLE_DEVICES=1


In [2]:
from xvla_wlr.agent import XVLAAgent, XVLAAction, XVLAObservation, XVLA_DOMAIN_IDS


In [3]:
# import accelerate

# accelerator = accelerate.Accelerator(
#     mixed_precision="bf16",
#     dynamo_plugin=accelerate.utils.TorchDynamoPlugin(
#         backend="inductor",
#         mode="reduce-overhead",
#         fullgraph=True,
#         # dynamic=True,
#     )
# )

In [3]:
import torch

torch.set_float32_matmul_precision("high")

agent = XVLAAgent(
    "/home/ace/X-VLA/workspaces/experiment_sample/checkpoints/current/checkpoint.json",
    accelerator=True,
    dtype=torch.bfloat16,
)

FileNotFoundError: [Errno 2] No such file or directory: '/home/ace/X-VLA/workspaces/experiment_sample/checkpoints/current/checkpoint.json'

In [5]:

torch.get_float32_matmul_precision()

'high'

In [13]:
agent._model.active_adapters()

['default']

In [14]:
agent._model

XVLA(
  (action_space): EE6DActionSpace(
    (mse): MSELoss()
    (bce): BCEWithLogitsLoss()
  )
  (vlm): Florence2ForConditionalGeneration(
    (vision_tower): DaViT(
      (convs): ModuleList(
        (0): ConvEmbed(
          (proj): Conv2d(3, 256, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        )
        (1): ConvEmbed(
          (proj): Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        )
        (2): ConvEmbed(
          (proj): Conv2d(512, 1024, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        )
        (3): ConvEmbed(
          (proj): Conv2d(1024, 2048, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        )
      )
      (blocks): ModuleList(

In [8]:
# %%timeit -n 10
import torch

# compute_actions = torch.compile(agent.compute_actions, mode="max-autotune", disable=False)
compute_actions = agent.compute_actions
# %timeit -n 10 
compute_actions(XVLAObservation.sample())

XVLAAction(ee_transforms=tensor([[[[[-0.4551,  0.8477,  0.2656, -0.3359],
           [ 0.7305,  0.5273, -0.4414, -0.2344],
           [-0.5156, -0.0068, -0.8594, -0.2393],
           [ 0.0000,  0.0000,  0.0000,  1.0000]],

          [[-0.6797, -0.6367,  0.3594,  0.3047],
           [-0.5391,  0.0996, -0.8398,  0.4375],
           [ 0.5000, -0.7656, -0.4121, -0.4004],
           [ 0.0000,  0.0000,  0.0000,  1.0000]]],


         [[[-0.4570,  0.8555,  0.2559, -0.3340],
           [ 0.7266,  0.5234, -0.4414, -0.2334],
           [-0.5117, -0.0166, -0.8594, -0.2441],
           [ 0.0000,  0.0000,  0.0000,  1.0000]],

          [[-0.6641, -0.6406,  0.3848,  0.3047],
           [-0.5547,  0.0811, -0.8281,  0.4512],
           [ 0.5000, -0.7656, -0.4102, -0.4004],
           [ 0.0000,  0.0000,  0.0000,  1.0000]]],


         [[[-0.4375,  0.8672,  0.2539, -0.3281],
           [ 0.7461,  0.5078, -0.4375, -0.2197],
           [-0.5078, -0.0020, -0.8711, -0.2432],
           [ 0.0000,  0.0000,  0

In [7]:
compute_actions(XVLAObservation.sample())

XVLAAction(ee_transforms=tensor([[[[[-0.4551,  0.8477,  0.2656, -0.3359],
           [ 0.7305,  0.5273, -0.4414, -0.2344],
           [-0.5156, -0.0068, -0.8594, -0.2393],
           [ 0.0000,  0.0000,  0.0000,  1.0000]],

          [[-0.6797, -0.6406,  0.3574,  0.3066],
           [-0.5352,  0.1016, -0.8398,  0.4375],
           [ 0.5039, -0.7617, -0.4121, -0.4004],
           [ 0.0000,  0.0000,  0.0000,  1.0000]]],


         [[[-0.4570,  0.8516,  0.2559, -0.3340],
           [ 0.7266,  0.5234, -0.4414, -0.2334],
           [-0.5078, -0.0166, -0.8555, -0.2451],
           [ 0.0000,  0.0000,  0.0000,  1.0000]],

          [[-0.6641, -0.6406,  0.3848,  0.3027],
           [-0.5547,  0.0801, -0.8281,  0.4512],
           [ 0.5000, -0.7656, -0.4082, -0.4004],
           [ 0.0000,  0.0000,  0.0000,  1.0000]]],


         [[[-0.4375,  0.8633,  0.2539, -0.3281],
           [ 0.7422,  0.5078, -0.4375, -0.2207],
           [-0.5078, -0.0029, -0.8633, -0.2432],
           [ 0.0000,  0.0000,  0

In [10]:
%timeit -n 10 compute_actions(XVLAObservation.sample())


36.9 ms ± 708 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [12]:
import torch

compute_actions = torch.compile(agent.compute_actions, mode="max-autotune", disable=True)
with torch.profiler.profile() as prof:
    compute_actions(XVLAObservation.sample())
prof.export_chrome_trace("./logs/trace-v7.json")

In [ ]:
# accelerator.autocast?

Signature: accelerator.autocast(autocast_handler: 'AutocastKwargs' = None)
Docstring:
Will apply automatic mixed-precision inside the block inside this context manager, if it is enabled. Nothing
different will happen otherwise.

A different `autocast_handler` can be passed in to override the one set in the `Accelerator` object. This is
useful in blocks under `autocast` where you want to revert to fp32.

Example:

```python
>>> from accelerate import Accelerator

>>> accelerator = Accelerator(mixed_precision="fp16")
>>> with accelerator.autocast():
...     train()
```
File:      ~/X-VLA/.conda/lib/python3.11/site-packages/accelerate/accelerator.py
Type:      method

In [9]:
%%timeit -n 1

import torch

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    agent.compute_actions(XVLAObservation.sample())


105 ms ± 5.96 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
%%timeit -n 1
agent.compute_actions(XVLAObservation.sample())

137 ms ± 1.83 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
